# Experiment 1 — statistical testing

Full methodology of **Demšar (2006)** and the all-pairwise extension of **García & Herrera (2008)**:

* average ranks, **Friedman** test + **Iman–Davenport** correction (omnibus)
* **Nemenyi** post-hoc + **critical-difference diagrams**
* **Bonferroni–Dunn** and step procedures vs a control (Holm, Hochberg, Hommel, Holland, Finner, Li)
* all-pairwise adjusted p-values: Nemenyi, Holm, **Shaffer static**, **Bergmann–Hommel**
* **Wilcoxon signed-ranks** + **sign test** for the head-to-head pair
* win/loss/tie matrix and **PAMA**

Tests treat datasets as samples (one fold-mean per dataset per method), NO_HPO results.
All logic: `src/utils/statistical_testing.py`. Figures → `figures/experiment1/stats/`.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.paths import results_root
from src.visualizations.experiment_plots import (
    apply_style, reset_figure_dir, load_summary,
    performance_heatmap, method_ranking_bars, per_dataset_bars,
    learning_curve, imbalance_curve, metric_boxplots,
    hpo_improvement_bars, runtime_performance_scatter,
)
apply_style()

RESULTS_ROOT = results_root()          # override here if your copy lives elsewhere
SUMMARY_DIR  = RESULTS_ROOT / 'summaries'
FIGURES_DIR  = reset_figure_dir(PROJECT_ROOT / 'figures' / 'experiment1/stats')
print('results:', RESULTS_ROOT, '| figures:', FIGURES_DIR)

In [ ]:
import pandas as _pd
from src.utils import statistical_testing as st
df_pd  = load_summary(SUMMARY_DIR, experiment='experiment1', task='pd')
try:
    df_lgd = load_summary(SUMMARY_DIR, experiment='experiment1', task='lgd')
except FileNotFoundError:
    df_lgd = None

## PD — AUC (higher is better)

In [ ]:
mat = st.metric_matrix(df_pd, 'AUC')
ranks = st.average_ranks(mat)
print(st.friedman_test(mat))
display(ranks.to_frame('mean rank'))

In [ ]:
cd = st.nemenyi_cd(k=mat.shape[1], N=mat.shape[0])
st.plot_cd_diagram(ranks, cd, title=f'PD AUC — Nemenyi CD diagram (N={mat.shape[0]})',
                   out_path=FIGURES_DIR / 'pd_cd_diagram_auc.pdf')

In [ ]:
# One-vs-control (control = best-ranked): Bonferroni-Dunn + step procedures
print('Bonferroni-Dunn CD:', st.bonferroni_dunn_cd(mat.shape[1], mat.shape[0]))
display(st.control_apv_table(mat))

In [ ]:
# All-pairwise APVs (Garcia & Herrera 2008). Bergmann-Hommel auto-skips for k > 8.
apv = st.pairwise_apv_table(mat)
display(apv.head(25))
st.plot_significance_matrix(apv, mat.columns, procedure='shaffer',
                            out_path=FIGURES_DIR / 'pd_significance_shaffer_auc.pdf')

In [ ]:
# Head-to-head: top-2 ranked methods (Demsar two-classifier tests)
m1, m2 = ranks.index[0], ranks.index[1]
print(m1, 'vs', m2)
print('Wilcoxon:', st.wilcoxon_signed_rank(mat[m1], mat[m2]))
print('Sign    :', st.sign_test(mat[m1], mat[m2]))

In [ ]:
st.plot_win_loss_matrix(mat, out_path=FIGURES_DIR / 'pd_win_loss_auc.pdf')
st.plot_pama_bars(mat, metric_name='AUC', out_path=FIGURES_DIR / 'pd_pama_auc.pdf')
display(st.wlt_summary(mat)); display(st.pama(mat))

## PD — calibration (Brier, lower is better)

In [ ]:
mat_b = st.metric_matrix(df_pd, 'Brier')
ranks_b = st.average_ranks(mat_b, higher_is_better=False)
print(st.friedman_test(mat_b, higher_is_better=False))
st.plot_cd_diagram(ranks_b, st.nemenyi_cd(mat_b.shape[1], mat_b.shape[0]),
                   title='PD Brier — Nemenyi CD diagram',
                   out_path=FIGURES_DIR / 'pd_cd_diagram_brier.pdf')

## LGD — R² (higher is better)

In [ ]:
if df_lgd is not None:
    mat_r = st.metric_matrix(df_lgd, 'R2')
    ranks_r = st.average_ranks(mat_r)
    print(st.friedman_test(mat_r))
    st.plot_cd_diagram(ranks_r, st.nemenyi_cd(mat_r.shape[1], mat_r.shape[0]),
                       title=f'LGD R2 — Nemenyi CD diagram (N={mat_r.shape[0]})',
                       out_path=FIGURES_DIR / 'lgd_cd_diagram_r2.pdf')
    display(st.control_apv_table(mat_r))
    apv_r = st.pairwise_apv_table(mat_r)
    st.plot_significance_matrix(apv_r, mat_r.columns, procedure='shaffer',
                                out_path=FIGURES_DIR / 'lgd_significance_shaffer_r2.pdf')
    st.plot_win_loss_matrix(mat_r, higher_is_better=True,
                            out_path=FIGURES_DIR / 'lgd_win_loss_r2.pdf')

> **Reading the CD diagram:** methods joined by a bold bar are NOT significantly different at α=0.05 (Nemenyi). The Friedman/Iman–Davenport p decides whether post-hocs are warranted at all; Shaffer/Bergmann–Hommel APVs are the rigorous all-pairwise evidence (García & Herrera 2008).